In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

import os

load_dotenv()

# 从环境提取 API 密钥和基础 URL
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_API_URL")

# 用 ChatOpenAI 包装通义千问的 OpenAI 兼容端点
# create_agent 需要的是一个具备 bind_tools 方法的“模型实例”，
# 而不是 openai.chat.completions.create(...) 返回的 ChatCompletion 结果对象
model = ChatOpenAI(
    model="qwen3.7-flash",
    api_key=api_key,
    base_url=base_url,
    model_kwargs={"extra_body": {"enable_thinking": False}},
)

# 定义获取天气的函数（作为工具）
def get_weather(city: str) -> str:
    """获取天气信息"""
    return f"今天{city}天气晴朗，气温35摄氏度，湿度10%，风速2米/秒，主要紫外线"

# 定义系统提示
system_prompt = "你是一个天气助手，你可以根据用户的问题，提供天气信息。"

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=system_prompt,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "西安天气"}]}
)

# result["messages"] 是消息列表，最后一条是模型最终回复
print(result["messages"][-1].content)
